In [ ]:
!pip install -q pydicom open3d torchio pgzip

In [ ]:
import pandas as pd
import numpy as np
import os 
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import open3d as o3d
from pydicom import dcmread
import torch.nn.functional as F

from torch.utils.data import Dataset ,DataLoader
from torch.utils.data import Dataset
import torch
import torchio as tio
import glob
import cv2
import math , copy ,pgzip
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

In [ ]:
data_path = "/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/"

In [ ]:
train_csv = pd.read_csv("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train.csv")
# train_csv.head()

In [ ]:
train_series_descriptions = pd.read_csv("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_series_descriptions.csv")
# train_series_descriptions.head()

In [ ]:
train_label_coordinates = pd.read_csv("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_label_coordinates.csv")
# train_label_coordinates.head(30)

In [ ]:
train_label_coordinates["condition"].value_counts()

In [ ]:
condition = {"Sagittal T2/STIR": ["Spinal Canal Stenosis"], 
             "Axial T2": ["Left Subarticular Stenosis", "Right Subarticular Stenosis"], 
             "Sagittal T1": ["Left Neural Foraminal Narrowing", "Right Neural Foraminal Narrowing"],}

In [ ]:
LABEL_MAP = {'normal_mild': 0, 'moderate': 1, 'severe': 2}

In [ ]:
def retrieve_coordinate_training_data(train_path):
    def reshape_row(row):
        data = {'study_id': [], 'condition': [], 'level': [], 'severity': []}

        for column, value in row.items():
            if column not in ['study_id', 'series_id', 'instance_number', 'x', 'y', 'series_description']:
                parts = column.split('_')
                condition = ' '.join([word.capitalize() for word in parts[:-2]])
                level = parts[-2].capitalize() + '/' + parts[-1].capitalize()
                data['study_id'].append(row['study_id'])
                data['condition'].append(condition)
                data['level'].append(level)
                data['severity'].append(value)

        return pd.DataFrame(data)

    train = pd.read_csv(train_path + 'train.csv')
    label = pd.read_csv(train_path + 'train_label_coordinates.csv')
    train_desc = pd.read_csv(train_path + 'train_series_descriptions.csv')
    test_desc = pd.read_csv(train_path + 'test_series_descriptions.csv')
    sub = pd.read_csv(train_path + 'sample_submission.csv')

    new_train_df = pd.concat([reshape_row(row) for _, row in train.iterrows()], ignore_index=True)
    merged_df = pd.merge(new_train_df, label, on=['study_id', 'condition', 'level'], how='inner')
    final_merged_df = pd.merge(merged_df, train_desc, on=['series_id', 'study_id'], how='inner')
    final_merged_df['severity'] = final_merged_df['severity'].map(
        {'Normal/Mild': 'normal_mild', 'Moderate': 'moderate', 'Severe': 'severe'})

    final_merged_df['row_id'] = (
            final_merged_df['study_id'].astype(str) + '_' +
            final_merged_df['condition'].str.lower().str.replace(' ', '_') + '_' +
            final_merged_df['level'].str.lower().str.replace('/', '_')
    )

    # Create the image_path column
    final_merged_df['image_path'] = (
            f'{train_path}/train_images/' +
            final_merged_df['study_id'].astype(str) + '/' +
            final_merged_df['series_id'].astype(str) + '/' +
            final_merged_df['instance_number'].astype(str) + '.dcm'
    )

    return final_merged_df

In [ ]:
train_data = retrieve_coordinate_training_data(data_path)

In [ ]:
def dicom_to_pcd(dir_path,series_type_dict=None,resize_slice=True,resize_method="nearest",
                 img_size=(128,128),downsampling_factor=1
                ,stack_slices_thickness=True,series_dsc=None):
    pcd_overall = o3d.geometry.PointCloud()
    for path in glob.glob(os.path.join(dir_path,"**/*.dcm"),recursive=True):
        dicom_slices = dcmread(path)
        series_id = os.path.basename(os.path.dirname(path))
        study_id = os.path.basename(os.path.dirname(os.path.dirname(path)))
        if series_type_dict is None or int(series_id) not in series_type_dict:
            series_dsc = dicom_slices.SeriesDescription
        else:
            series_dsc = series_type_dict[int(series_dsc)]
            series_dsc = series_type_dict.split(" ")[-1]
        
        x_orig , y_orig = dicom_slices.pixel_array.shape
        if resize_slice:
            if resize_method == "nearest":
                img = np.expand_dims(cv2.resize(dicom_slices.pixel_array,img_size,interpolation=cv2.INTER_AREA),-1)
            elif resize_method == "maxpool":
                img_tensor = torch.tensor(dicom_slices.pixel_array).float()
                img = F.adaptive_max_pool2d(img_tensor.unsqueeze(0),img_size).numpy()
            else:
                print("invalid resize method")
        else:
            img = np.expand_dims(np.array(dicom_slices.pixel_array),-1)
        x,y,z = np.where(img)
        downsampling_factor_iter = max(downsampling_factor,int(math.ceil(len(x)/6e6)))
        index_voxel = np.vstack((x,y,z))[:,::downsampling_factor_iter]
        grid_index_array = index_voxel.T
        pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(grid_index_array.astype(np.float64)))
        vals = np.expand_dims(img[x,y,z][::downsampling_factor_iter],-1)
        #vals = vals/np.max(vals)
        if series_dsc == "T1":
            vals = np.pad(vals,((0,0),(0,2)))
        elif series_dsc == "T2":
            vals = np.pad(vals,((0,0),(1,1)))
        elif series_dsc == "T2/STIR":
            vals = np.pad(vals,((0,0),(2,0)))   
        else:
            vals = np.pad(vals, ((0, 0), (0, 2)), constant_values=-1)
        
        vals = vals.astype(np.float64) / 255.0
        
        
        if vals.shape[1] != 3:
            vals = vals.reshape(-1, 3)
        pcd.colors = o3d.utility.Vector3dVector(vals.astype(np.float64))
        if resize_slice:
            transform_matrix_F = np.matrix([[0,y_orig/img_size[1],0,0],
                                         [x_orig/img_size[0],0,0,0],
                                         [0,0,1,0],
                                         [0,0,0,1]])
        else:
            transform_matrix_F = np.matrix([0,1,0,0],   #permutate rows and coloumsn
                                        [1,0,0,0],
                                        [0,0,1,0],
                                        [0,0,0,1])
        dx , dy = dicom_slices.PixelSpacing
        dz = dicom_slices.SliceThickness
        
        X = np.array(list(dicom_slices.ImageOrientationPatient[:3])+[0]) * dx
        Y = np.array(list(dicom_slices.ImageOrientationPatient[3:])+[0]) * dy
        S = np.array(list(dicom_slices.ImagePositionPatient)+[1])
        transform_matrxi = np.array([X,Y,np.zeros(len(X)),S]).T
        transform_matrix = transform_matrxi @ transform_matrix_F
        if stack_slices_thickness:
            for z in range(int(dz)):
                pos = list(dicom_slices.ImagePositionPatient)
                if series_dsc =="T2":
                    pos[-1] += z
                else:
                    pos[0] += z
                S = np.array(pos +[1])
                transform_matrxi = np.array([X,Y,np.zeros(len(X)),S]).T
                transform_matrix = transform_matrxi @ transform_matrix_F
                pcd_overall += copy.deepcopy(pcd).transform(transform_matrix)
        else:
            pcd_overall += copy.deepcopy(pcd).transform(transform_matrix)
    return pcd_overall

def read_study_voxel_grid(dir_path,study_id,img_size=(128,128),chaching=True,chace_base_path="/kaggle/working/3d_chache",downsampling_factor=1,
                          series_type_dict=None):
    if chaching:
        os.makedirs(os.path.join(chace_base_path,str(study_id)),exist_ok=True)
        cahe_path = os.path.join(chace_base_path,str(study_id),f"cached_grid_v2_{img_size[0]}.npy.gz")
        f = None
        if os.path.exists(cahe_path):
            try:
                f = pgzip.PgzipFile(cahe_path,"r")
                ret = np.load(f,allow_pickle=True)
                f.close()
                return ret
            except Exception as e:
                print(dir_path,"\n",e)
                if f:
                    f.close()
                os.remove(cahe_path)
        
    pcd_overall = dicom_to_pcd(dir_path,series_type_dict=series_type_dict,downsampling_factor=downsampling_factor,img_size=img_size)
    if pcd_overall is None:
        raise ValueError(f"Failed to create point cloud for study ID {study_id}. Check if DICOM files are correctly processed.")

    box = pcd_overall.get_axis_aligned_bounding_box()
    max_b = np.array(box.get_max_bound())
    min_b = np.array(box.get_min_bound())
    pts = (np.array(pcd_overall.points)-min_b) * (img_size[0]-1,img_size[0]-1,img_size[0]-1) / (max_b - min_b)
    coords = np.round(pts).astype(np.int32)
    vals = np.array(pcd_overall.colors,dtype=np.float16)
    grid = np.zeros((3,img_size[0],img_size[0],img_size[0]),dtype=np.float16)
    indicies = coords[:,0] , coords[:,1] , coords[:,2]
    
    np.maximum.at(grid[0],indicies,vals[:,0])
    np.maximum.at(grid[1],indicies,vals[:,1])
    np.maximum.at(grid[2],indicies,vals[:,2])
    
    if chaching:
        f = pgzip.PgzipFile(cahe_path,"w")
        np.save(f,grid)
        f.close()
    return grid

# Test Visual

In [ ]:
def dicom_to_pcd(dir_path, series_type_dict=None, resize_slice=True, resize_method="nearest",
                 img_size=(128, 128), downsampling_factor=1, stack_slices_thickness=True, series_dsc=None):
    pcd_overall = o3d.geometry.PointCloud()
    count = 0  # Biến đếm số file đã xử lý
    for path in glob.glob(os.path.join(dir_path, "**/*.dcm"), recursive=True):
        if count >= 3:  # Dừng sau khi xử lý 3 file
            break
        dicom_slices = dcmread(path)
        series_id = os.path.basename(os.path.dirname(path))
        study_id = os.path.basename(os.path.dirname(os.path.dirname(path)))

        # Xử lý tương tự code gốc
        if series_type_dict is None or int(series_id) not in series_type_dict:
            series_dsc = dicom_slices.SeriesDescription
        else:
            series_dsc = series_type_dict[int(series_dsc)]
            series_dsc = series_type_dict.split(" ")[-1]

        x_orig, y_orig = dicom_slices.pixel_array.shape
        if resize_slice:
            if resize_method == "nearest":
                img = np.expand_dims(cv2.resize(dicom_slices.pixel_array, img_size, interpolation=cv2.INTER_AREA), -1)
            elif resize_method == "maxpool":
                img_tensor = torch.tensor(dicom_slices.pixel_array).float()
                img = F.adaptive_max_pool2d(img_tensor.unsqueeze(0), img_size).numpy()
            else:
                print("Invalid resize method")
        else:
            img = np.expand_dims(np.array(dicom_slices.pixel_array), -1)

        x, y, z = np.where(img)
        downsampling_factor_iter = max(downsampling_factor, int(math.ceil(len(x) / 6e6)))
        index_voxel = np.vstack((x, y, z))[:, ::downsampling_factor_iter]
        grid_index_array = index_voxel.T
        pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(grid_index_array.astype(np.float64)))
        vals = np.expand_dims(img[x, y, z][::downsampling_factor_iter], -1)

        # Mapping color based on series_dsc
        if series_dsc == "T1":
            vals = np.pad(vals, ((0, 0), (0, 2)))
        elif series_dsc == "T2":
            vals = np.pad(vals, ((0, 0), (1, 1)))
        elif series_dsc == "T2/STIR":
            vals = np.pad(vals, ((0, 0), (2, 0)))
        else:
            vals = np.pad(vals, ((0, 0), (0, 2)), constant_values=-1)

        vals = vals.astype(np.float64) / 255.0

        if vals.shape[1] != 3:
            vals = vals.reshape(-1, 3)
        pcd.colors = o3d.utility.Vector3dVector(vals.astype(np.float64))

        dx, dy = dicom_slices.PixelSpacing
        dz = dicom_slices.SliceThickness

        X = np.array(list(dicom_slices.ImageOrientationPatient[:3]) + [0]) * dx
        Y = np.array(list(dicom_slices.ImageOrientationPatient[3:]) + [0]) * dy
        S = np.array(list(dicom_slices.ImagePositionPatient) + [1])
        transform_matrxi = np.array([X, Y, np.zeros(len(X)), S]).T
        transform_matrix = transform_matrxi @ np.identity(4)

        pcd_overall += copy.deepcopy(pcd).transform(transform_matrix)
        count += 1  # Tăng biến đếm sau mỗi lần xử lý
    return pcd_overall


In [ ]:
import glob
import os
import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d

# Đường dẫn thư mục chứa file DICOM
dicom_dir = "/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_images/1039182563/1737958872"

# Lọc danh sách file DICOM
dicom_files = glob.glob(os.path.join(dicom_dir, "*.dcm"))
print(f"Total DICOM files: {len(dicom_files)}")

# Chọn 3 file đầu tiên
dicom_files = dicom_files[:3]
print(f"Selected DICOM files: {dicom_files}")

# Giả sử hàm dicom_to_pcd trả về point cloud
# Thay hàm này bằng hàm bạn đã định nghĩa
point_cloud = dicom_to_pcd(
    dir_path=dicom_dir,
    series_type_dict=None,
    resize_slice=True,
    resize_method="nearest",
    img_size=(128, 128),
    downsampling_factor=2,
    stack_slices_thickness=True
)

# Lưu kết quả Point Cloud vào file để kiểm tra sau
output_file = "output_point_cloud.ply"
o3d.io.write_point_cloud(output_file, point_cloud)
print(f"Point cloud saved to {output_file}")

# Chuyển đổi dữ liệu Point Cloud sang định dạng NumPy
points = np.asarray(point_cloud.points)
colors = np.asarray(point_cloud.colors)

# Kiểm tra và chuẩn hóa giá trị màu sắc nếu cần
if colors.max() > 1:
    colors = colors / 255.0  # Chuẩn hóa về phạm vi [0, 1]

# Kiểm tra phạm vi giá trị màu
# print(f"Colors range: min={colors.min()}, max={colors.max()}")

# Hiển thị Point Cloud bằng Matplotlib
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=colors, s=0.1)
ax.set_xlabel("X axis")
ax.set_ylabel("Y axis")
ax.set_zlabel("Z axis")
plt.show()


In [ ]:
import os
import numpy as np
import open3d as o3d
from pydicom import dcmread

# Đường dẫn đến file .dcm
dcm_file_path = "/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_images/1039182563/1737958872/14.dcm"

# Đọc file .dcm
dicom_slices = dcmread(dcm_file_path)

# Chuyển đổi file .dcm thành point cloud
pcd_overall = dicom_to_pcd(os.path.dirname(dcm_file_path), resize_slice=True, img_size=(128, 128))

# Hiển thị point cloud
o3d.visualization.draw_geometries([pcd_overall])

In [ ]:
# Hiển thị mặt cắt 2D (để kiểm tra hình ảnh gốc): Bạn cũng có thể kiểm tra dữ liệu DICOM gốc:

import pydicom
import matplotlib.pyplot as plt

# Đọc một file DICOM để xem mặt cắt
dcm_path = "/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_images/1039182563/1737958872/12.dcm"
dcm_data = pydicom.dcmread(dcm_path)

# Hiển thị ảnh DICOM
plt.imshow(dcm_data.pixel_array, cmap="gray")
plt.title("DICOM Slice")
plt.axis("off")
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d
from matplotlib.colors import Normalize
from matplotlib.colorbar import ColorbarBase

# Đường dẫn dữ liệu DICOM
dicom_dir_path = "/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/"

# Hàm để hiển thị Point Cloud với Open3D
def visualize_point_cloud(dicom_dir, img_size=(128, 128)):
    print("Processing point cloud...")
    pcd_result = dicom_to_pcd(dicom_dir, img_size=img_size, downsampling_factor=2)
    if not pcd_result.has_points():
        raise ValueError("No points found in the processed Point Cloud.")
    
    # Lưu và hiển thị hình ảnh Point Cloud
    output_path = "point_cloud.png"
    vis = o3d.visualization.Visualizer()
    vis.create_window(visible=False)
    vis.add_geometry(pcd_result)
    vis.poll_events()
    vis.update_renderer()
    vis.capture_screen_image(output_path)
    vis.destroy_window()
    
    print(f"Point Cloud visualization saved to {output_path}.")
    return output_path

# Hàm để hiển thị Voxel Grid
def visualize_voxel_grid(dicom_dir, study_id, img_size=(128, 128)):
    print("Processing voxel grid...")
    voxel_grid = read_study_voxel_grid(dicom_dir, study_id, img_size=img_size)
    
    # Hiển thị một số lát cắt
    num_slices = voxel_grid.shape[1]  # Tổng số lát
    slices_to_show = [num_slices // 4, num_slices // 2, 3 * num_slices // 4]  # Chọn 3 lát cắt

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    cmap = 'gray'
    
    for ax, slice_idx in zip(axes, slices_to_show):
        img = voxel_grid[0, slice_idx, :, :]  # Lấy kênh đầu tiên (T1)
        ax.imshow(img, cmap=cmap)
        ax.set_title(f"Slice {slice_idx}")
        ax.axis('off')

    plt.tight_layout()
    output_path = "voxel_grid_slices.png"
    plt.savefig(output_path, dpi=300)
    print(f"Voxel grid slices visualization saved to {output_path}.")
    return output_path

# Thực thi và hiển thị kết quả
point_cloud_image = visualize_point_cloud(dicom_dir_path, img_size=(128, 128))
voxel_grid_image = visualize_voxel_grid(dicom_dir_path, study_id="4003253", img_size=(128, 128))

print("Visualization complete.")

In [ ]:
class studyleveldataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, base_path, transform_3d, vol_size=(128, 128, 128), is_mirror=False):
        self.base_path = base_path
        self.is_mirror = is_mirror
        self.dataframe = dataframe[["study_id", "series_id", "series_description", "condition", "severity", "level"]].drop_duplicates()
        self.subjects = self.dataframe[["study_id"]].drop_duplicates().reset_index()
        self.series = self.dataframe[["study_id", "series_id"]].drop_duplicates().groupby("study_id")["series_id"].apply(list).to_dict()
        self.series_descs = {e[0]: e[1] for e in self.dataframe[["study_id", "series_description"]].drop_duplicates().values}
        self.transforms_3d = transform_3d
        self.levels = sorted(self.dataframe["level"].unique())
        self.labels = self._get_labels()
        self.vol_size = vol_size

    def __len__(self):
        return len(self.subjects) * (2 if self.is_mirror else 1)

    def __getitem__(self, index):
        is_mirror = index >= len(self.subjects)
        curr = self.subjects.iloc[index % len(self.subjects)]
        label_array = np.array(self.labels[curr["study_id"]])
        study_path = os.path.join(self.base_path, str(curr["study_id"]))
        study_images = read_study_voxel_grid(study_path, curr["study_id"], series_type_dict=self.series_descs)
        
        # Padding the label array
        max_length = 25  
        label_array = np.pad(label_array, (3, max(0, max_length - len(label_array))), 'constant', constant_values=3)  # Padding with '3'

        # Mirror operation on the label array
        if is_mirror:
            half_length = len(label_array) // 2
            temp = label_array[:half_length].copy()
            label_array[:half_length] = label_array[half_length:].copy()
            label_array[half_length:] = temp

        # Apply majority voting to get a single label
        label = self.apply_majority_voting(label_array)
        
        # Apply 3D transformations if provided
        if self.transforms_3d is not None:
            study_images = torch.FloatTensor(study_images)
            if is_mirror:
                study_images = torch.flip(study_images, [1])
            study_images = self.transforms_3d(study_images)
            return study_images.to(torch.half), torch.tensor(label, dtype=torch.long)
        
        return torch.FloatTensor(study_images.copy()), torch.tensor(label, dtype=torch.long)

    def _get_labels(self):
        labels = dict()
        for study_id, group in self.dataframe.groupby("study_id"):
            group = group[["condition", "level", "severity"]].drop_duplicates().sort_values(["condition", "level"])
            label_indices = []
            for _, row in group.iterrows():
                if row["severity"] in LABEL_MAP:
                    label_indices.append(LABEL_MAP[row["severity"]])
                else:
                    label_indices.append(3)  # Default to padding value if severity is missing
            labels[study_id] = label_indices
        return labels

    def apply_majority_voting(self, label_array):
        valid_labels = label_array[label_array != 3]  # Exclude padding
        if len(valid_labels) == 0:
            print("d")
            # If all labels are padding, return a default value (e.g., the most frequent class 0)
            return 0
        
        return np.bincount(valid_labels).argmax()  # Majority voting on valid labels only

In [ ]:
def create_dataset_and_loaders_kfolds(df, base_path, transform_train=None, transform_val=None, batch_size=16, split_k=5, num_workers=0):
    """
Create datasets and data loaders using K-fold cross-validation.

Parameters:
    df (pd.DataFrame): DataFrame containing the dataset information.
    base_path (str): Path to the base directory of the dataset.
    transform_train (callable, optional): Transformations for training data.
    transform_val (callable, optional): Transformations for validation data.
    batch_size (int): Batch size for DataLoader.
    split_k (int): Number of folds for K-fold cross-validation.

Returns:
    list: List of tuples containing train_loader, val_loader, train_dataset, and val_dataset for each fold.
"""
    df = df.dropna()
    np.random.seed(42)
    study_ids = df["study_id"].unique()
    np.random.shuffle(study_ids)
    folds = np.array_split(study_ids, split_k) 
    loaders = []
    
    for fold in folds:
        val_df = df[df["study_id"].isin(fold)]
        train_df = df[~df["study_id"].isin(fold)]
        
        train_df.reset_index(drop=True, inplace=True)
        val_df.reset_index(drop=True, inplace=True)
        
        # Calculate 50% of the dataset sizes
        train_size = int(len(train_df) * 0.2)
        val_size = int(len(val_df) * 0.2)
        
        # Limit the datasets to 50%
        train_df = train_df.iloc[:train_size]
        val_df = val_df.iloc[:val_size]
        
        train_dataset = studyleveldataset(train_df, base_path, transform_3d=transform_train)
        val_dataset = studyleveldataset(val_df, base_path, transform_3d=transform_val)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
        
        loaders.append((train_loader, val_loader, train_dataset, val_dataset))
    
    return loaders

In [ ]:
transform = tio.Compose([tio.RescaleIntensity(out_min_max=(0,1))])
folds = create_dataset_and_loaders_kfolds(train_data,base_path=os.path.join(data_path,"train_images"))

In [ ]:
train_loader,val_loader,train_dataset,val_dataset=folds[0]

In [ ]:
study_images , labels = next(iter(train_loader))
print(study_images.shape)
print(labels.shape)
print(labels)

In [ ]:
class PatchEmbed3D(nn.Module):
    def __init__(self, img_size, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 3
        self.proj = nn.Conv3d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)  # (batch_size, embed_dim, num_patches)
        x = x.transpose(1, 2)  # (batch_size, num_patches, embed_dim)
        return x

class Attention3D(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_dropout = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_dropout(x)

        return x

class TransformerBlock3D(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, drop=0., attn_drop=0., act_layer=nn.GELU):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention3D(dim, num_heads, qkv_bias, attn_drop, drop)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            act_layer(),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden_dim, dim),
            nn.Dropout(drop)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class visiontransformer3D(nn.Module):
    def __init__(self,img_size=128,patch_size=16,in_channels=3,num_class=3,embed_dim=768,depth=12,num_heads=3,mlp_ratio=4.,qkv_bias=0,drop_rate=0,attn_drop_rate=0.):
        super().__init__()
        self.patch_embed = PatchEmbed3D(img_size=img_size)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1,1,embed_dim))
        self.pos_token = nn.Parameter(torch.zeros(1,num_patches+1,embed_dim))
        self.pos_drop = nn.Dropout(p=drop_rate)
        self.blocks = nn.ModuleList([
            TransformerBlock3D(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, drop=drop_rate, attn_drop=attn_drop_rate)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, 3 )  

        nn.init.trunc_normal_(self.pos_token, std=.02)
        nn.init.trunc_normal_(self.cls_token, std=.02)
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
    def forward(self,x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_token = self.cls_token.expand(B,-1,-1)
        x = torch.cat((cls_token,x),dim=1)
        
        x = x + self.pos_token
        x = self.pos_drop(x)
        
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        x = x[:,0]
        x = self.head(x)
        x=F.softmax(x, dim=-1)
        x = x.view(B,3)
        return x

In [ ]:
model = visiontransformer3D()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
from tqdm import tqdm
import os  # For managing file paths

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, device='cpu', fold=1):

    best_val_loss = float("inf")  # To track the best validation loss
    # Lists to store metrics
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", unit="batch")
        
        for batch_idx, (images, labels) in enumerate(train_pbar):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            # Update batch statistics
            _, predicted = torch.max(outputs.data, 1)
            batch_correct = (predicted == labels).sum().item()
            batch_total = labels.size(0)
            
            # Update epoch statistics
            epoch_loss += loss.item()
            epoch_correct += batch_correct
            epoch_total += batch_total
            
            # Update progress bar with current batch statistics
            batch_acc = batch_correct / batch_total
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{batch_acc:.4f}'
            })
        
        # Calculate epoch statistics
        epoch_avg_loss = epoch_loss / len(train_loader)
        epoch_accuracy = epoch_correct / epoch_total

        train_losses.append(epoch_avg_loss)
        train_accuracies.append(epoch_accuracy)
        
        print(f"\nEpoch {epoch + 1} Summary:")
        print(f"Training Loss: {epoch_avg_loss:.4f}, Training Accuracy: {epoch_accuracy:.4f}")
        
        # Validation phase
        val_avg_loss, val_accuracy = validate_model(model, val_loader, criterion, device)
        val_losses.append(val_avg_loss)
        val_accuracies.append(val_accuracy)
        
    print("Training completed.")
    return train_losses, train_accuracies, val_losses , val_accuracies 

def validate_model(model, val_loader, criterion, device='cpu'):
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    # Add tqdm for validation
    val_pbar = tqdm(val_loader, desc="Validation", unit="batch")
    
    with torch.no_grad():
        for images, labels in val_pbar:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Update validation statistics
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            batch_correct = (predicted == labels).sum().item()
            batch_total = labels.size(0)
            
            val_correct += batch_correct
            val_total += batch_total
            
            # Update validation progress bar
            batch_acc = batch_correct / batch_total
            val_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{batch_acc:.4f}'
            })
    
    # Calculate final validation metrics
    val_avg_loss = val_loss / len(val_loader)
    val_accuracy = val_correct / val_total
    
    print(f"\nValidation Summary:")
    print(f"Validation Loss: {val_avg_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")
    return val_avg_loss, val_accuracy

if __name__ == "__main__":
    # Assuming you have your dataset and dataloaders ready
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = visiontransformer3D().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for fold, (train_loader, val_loader, train_dataset, val_dataset) in enumerate(folds, 1):
        print(f"\nStarting training for fold {fold}")
        print("=" * 50)
        train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, device=device, fold=fold)
        print("=" * 50)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.title("Loss over Epochs")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

# Plot accuracies
plt.figure(figsize=(10, 5))
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.title("Accuracy over Epochs")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()